In [1]:
# ================================
# 1. Import Libraries
# ================================
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib


# ================================
# 2. Load Segmented Data
# ================================
df = pd.read_csv("../model/segmented_customers.csv")
shap_importance = pd.read_csv("../model/shap_feature_importance.csv")

print("Data loaded successfully!")
print("Shape:", df.shape)
print("\nColumns available:", df.columns.tolist())


# ================================
# 3. Retention Strategy Function
# (Based on YOUR actual SHAP top features)
# ================================
def get_retention_strategy(customer):
    strategies  = []
    priority    = []

    # --- Based on SHAP Feature #1: Tenure Months ---
    if customer['Tenure Months'] < 0:  # below average tenure (scaled)
        strategies.append("🎁  Assign dedicated onboarding manager for first 6 months")
        strategies.append("📦  Offer welcome loyalty package — free month or service upgrade")
        priority.append("URGENT")

    # --- Based on SHAP Feature #2: Fiber Optic Internet ---
    if customer.get('Internet Service_Fiber optic', 0) == 1:
        strategies.append("📶  Offer fiber optic service quality review and speed guarantee")
        strategies.append("💰  Provide fiber loyalty discount — 10% off for 6 months")
        priority.append("HIGH")

    # --- Based on SHAP Feature #3: Monthly Charges ---
    if customer['Monthly Charges'] > 0:  # above average charges (scaled)
        strategies.append("💳  Offer customized bundle plan to reduce monthly bill")
        strategies.append("🎯  Provide loyalty discount — 15% off next 3 months")
        priority.append("HIGH")

    # --- Based on SHAP Feature #4: Dependents ---
    if customer.get('Dependents_Yes', 0) == 0:
        strategies.append("👨‍👩‍👧  Promote family/dependents plan benefits and discounts")
        priority.append("MEDIUM")

    # --- Based on SHAP Feature #5: Electronic Check ---
    if customer.get('Payment Method_Electronic check', 0) == 1:
        strategies.append("🏦  Incentivize switch to auto-pay — offer 5% monthly discount")
        strategies.append("💡  Send email campaign highlighting auto-pay convenience")
        priority.append("MEDIUM")

    # --- Based on SHAP Feature #6: Two Year Contract ---
    if customer.get('Contract_Two year', 0) == 0 and customer.get('Contract_One year', 0) == 0:
        strategies.append("📋  Offer discounted 1-year or 2-year contract upgrade")
        strategies.append("🔒  Highlight contract benefits — price lock and priority support")
        priority.append("HIGH")

    # --- Based on SHAP Feature #7: Multiple Lines ---
    if customer.get('Multiple Lines_Yes', 0) == 0:
        strategies.append("📱  Promote multi-line plan — offer second line at 50% off")
        priority.append("MEDIUM")

    # --- Based on SHAP Features #8-9: Streaming Services ---
    if customer.get('Streaming TV_Yes', 0) == 0 and customer.get('Streaming Movies_Yes', 0) == 0:
        strategies.append("🎬  Offer free 3-month streaming TV and movies trial")
        priority.append("LOW")

    # Default if no strategy triggered
    if not strategies:
        strategies.append("✅  Customer profile is stable — send quarterly satisfaction survey")
        priority.append("LOW")

    return strategies, priority


# ================================
# 4. Apply Strategy to All Customers
# ================================
print("\nApplying retention strategies...")

all_strategies = []
for idx, row in df.iterrows():
    strats, prios = get_retention_strategy(row)
    all_strategies.append({
        'Customer_Index'    : idx,
        'Risk_Segment'      : row['Risk_Segment'],
        'Churn_Probability' : round(row['Churn_Probability'], 4),
        'Strategies'        : " | ".join(strats),
        'Priority'          : prios[0] if prios else 'LOW'
    })

df_strategies = pd.DataFrame(all_strategies)
print(f"Strategies generated for {len(df_strategies)} customers")


# ================================
# 5. Strategy Summary
# ================================
print("\n--- Strategy Priority Distribution ---")
print(df_strategies['Priority'].value_counts())

print("\n--- Strategies by Risk Segment ---")
for segment in ['High Risk', 'Medium Risk', 'Low Risk']:
    seg_data = df_strategies[df_strategies['Risk_Segment'] == segment]
    print(f"\n{segment} ({len(seg_data)} customers):")
    print(f"  Most common strategies:")
    # Count individual strategies
    all_strats = []
    for s in seg_data['Strategies']:
        all_strats.extend(s.split(" | "))
    strat_counts = pd.Series(all_strats).value_counts().head(3)
    for strat, count in strat_counts.items():
        print(f"    {strat[:60]}... → {count} customers")


# ================================
# 6. Sample High Risk Strategies
# ================================
print("\n--- Sample: Top 5 High Risk Customers ---")
high_risk = df_strategies[
    df_strategies['Risk_Segment'] == 'High Risk'
].sort_values('Churn_Probability', ascending=False).head(5)

for _, row in high_risk.iterrows():
    print(f"\nCustomer {row['Customer_Index']} "
          f"(Churn Prob: {row['Churn_Probability']})")
    for s in row['Strategies'].split(" | "):
        print(f"  → {s}")


# ================================
# 7. Business Impact Summary
# ================================
print("\n--- Business Impact Summary ---")

high_risk_count  = len(df[df['Risk_Segment'] == 'High Risk'])
med_risk_count   = len(df[df['Risk_Segment'] == 'Medium Risk'])

# Estimated avg monthly revenue per customer (using scaled values — approximate)
avg_monthly      = 65  # approximate average from telecom dataset

potential_loss_high = high_risk_count * avg_monthly
potential_loss_med  = med_risk_count  * avg_monthly

print(f"  High Risk customers        : {high_risk_count}")
print(f"  Medium Risk customers      : {med_risk_count}")
print(f"  Est. monthly revenue at risk (High) : ${potential_loss_high:,}")
print(f"  Est. monthly revenue at risk (Med)  : ${potential_loss_med:,}")
print(f"  Total revenue at risk               : ${potential_loss_high + potential_loss_med:,}")


# ================================
# 8. Strategy Priority Plot
# ================================
print("\nGenerating strategy plots...")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Priority distribution
priority_counts = df_strategies['Priority'].value_counts()
priority_order  = ['URGENT', 'HIGH', 'MEDIUM', 'LOW']
priority_colors = ['#E24B4A', '#EF9F27', '#378ADD', '#1D9E75']
p_counts = [priority_counts.get(p, 0) for p in priority_order]

axes[0].bar(priority_order, p_counts, color=priority_colors,
            edgecolor='white', linewidth=0.5)
axes[0].set_title('Customers by Action Priority', fontsize=13)
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(p_counts):
    axes[0].text(i, v + 20, str(v), ha='center', fontsize=11, fontweight='bold')

# Segment vs Priority heatmap style
segment_priority = df_strategies.groupby(
    ['Risk_Segment', 'Priority']
).size().unstack(fill_value=0)

segment_priority.plot(
    kind='bar', ax=axes[1],
    color=priority_colors[:len(segment_priority.columns)],
    edgecolor='white', linewidth=0.5
)
axes[1].set_title('Priority Distribution per Segment', fontsize=13)
axes[1].set_ylabel('Number of Customers')
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend(title='Priority', fontsize=10)

plt.suptitle('Retention Strategy Overview', fontsize=15,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("../model/retention_strategy_overview.png",
            dpi=150, bbox_inches='tight')
plt.close()
print("Saved → retention_strategy_overview.png")


# ================================
# 9. Save Final Output
# ================================
df_strategies.to_csv("../model/retention_strategies.csv", index=False)
print("Saved → retention_strategies.csv")

print("\nRetention Strategy Phase Complete!")
print("Files saved:")
print("  retention_strategy_overview.png")
print("  retention_strategies.csv")

Data loaded successfully!
Shape: (7043, 33)

Columns available: ['Tenure Months', 'Monthly Charges', 'Total Charges', 'Gender_Male', 'Senior Citizen_Yes', 'Partner_Yes', 'Dependents_Yes', 'Phone Service_Yes', 'Multiple Lines_No phone service', 'Multiple Lines_Yes', 'Internet Service_Fiber optic', 'Internet Service_No', 'Online Security_No internet service', 'Online Security_Yes', 'Online Backup_No internet service', 'Online Backup_Yes', 'Device Protection_No internet service', 'Device Protection_Yes', 'Tech Support_No internet service', 'Tech Support_Yes', 'Streaming TV_No internet service', 'Streaming TV_Yes', 'Streaming Movies_No internet service', 'Streaming Movies_Yes', 'Contract_One year', 'Contract_Two year', 'Paperless Billing_Yes', 'Payment Method_Credit card (automatic)', 'Payment Method_Electronic check', 'Payment Method_Mailed check', 'Churn_Probability', 'Risk_Segment', 'Actual_Churn']

Applying retention strategies...
Strategies generated for 7043 customers

--- Strategy P